# Time Series

## DatetimeIndex


In [ ]:
# Start by importing the packages we use in this chapter.
# This isn't strictly necessary with Python in Excel, as they are
# already imported in the initialization script.
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
# This creates a DatetimeIndex based on a start timestamp,
# number of periods, and frequency ("D" = daily).
daily_index = pd.date_range("2020-02-28", periods=4, freq="D")
daily_index

In [ ]:
# This creates a DatetimeIndex based on a start and end timestamp.
# The frequency is set to "weekly on Sundays" ("W-SUN").
weekly_index = pd.date_range("2020-01-01", "2020-01-31", freq="W-SUN")
weekly_index

In [ ]:
# Create a DataFrame based on weekly_index. This could be
# the visitor count of a museum that only opens on Sundays.
pd.DataFrame({"visitors": [21, 15, 33, 34]}, index=weekly_index)

In [ ]:
# This cells differs from the Python in Excel version: instead of using the xl
# function to load the data from Excel tables, we're using pd.read_csv to load
# the data directly from CSV files.
stocks = []
for ticker in ["MSFT", "AAPL", "AMZN", "GOOGL"]:
    df = pd.read_csv(
        f"csv/{ticker}.csv",  # For a refresher about f-strings, see Chapter 3
        index_col="date",
    )
    stocks.append(df)

# Assign to individual variables to be in line with Python in Excel
msft, aapl, amzn, googl = stocks

In [ ]:
msft.info()

In [ ]:
# Convert the index of all DataFrames to a DatetimeIndex
for df in [msft, aapl, amzn, googl]:
    df.index = pd.to_datetime(df.index)

In [ ]:
msft.info()

In [ ]:
msft["volume"] = msft["volume"].astype("float")
msft["volume"].dtype

In [ ]:
msft = msft.sort_index()

In [ ]:
msft.index.date

In [ ]:
msft.loc["2017", "adj_close"]

In [ ]:
msft.loc["2000-03":"2001-02", "adj_close"].plot()

## Shifting and Percentage Changes


In [ ]:
msft_close = msft[["adj_close"]].copy()
msft_close.head()

In [ ]:
msft_close.shift(1).head()

In [ ]:
returns = np.log(msft_close / msft_close.shift(1))
returns = returns.rename(columns={"adj_close": "returns"})
returns.head()

In [ ]:
# Plot a histogram with the daily log returns.
# The argument "bins=90" sets the number of bars in the histogram.
returns.plot.hist(bins=90)

In [ ]:
simple_rets = msft_close.pct_change()
simple_rets = simple_rets.rename(columns={"adj_close": "simple rets"})
simple_rets.head()

## Rebasing and Correlation


In [ ]:
parts = []  # List to collect individual DataFrames
for df in [msft, aapl, amzn, googl]:
    ticker = df["ticker"].iloc[0]  # Get ticker name
    df = df[["adj_close"]]  # Select adj_close column
    df = df.rename(columns={"adj_close": ticker})  # Rename column
    parts.append(df)  # Append the DataFrame to the parts list

In [ ]:
# Combine the 4 DataFrames into a single DataFrame
adj_close = pd.concat(parts, axis=1)
adj_close

In [ ]:
adj_close = adj_close.dropna()
adj_close.info()

In [ ]:
# Use a sample from March 2015 - February 2016
adj_close_sample = adj_close.loc["2015-03":"2016-02", :]
rebased_prices = adj_close_sample / adj_close_sample.iloc[0, :] * 100
rebased_prices.head(2)

In [ ]:
# "style" allows us to differentiate the lines in a black/white print
rebased_prices.plot(style=["-", "--", "-.", ":"])

In [ ]:
# Correlation of daily log returns
returns = np.log(adj_close / adj_close.shift(1))
correlations = returns.corr()
correlations

In [ ]:
# Use seaborn to create a heatmap
ax = sns.heatmap(
    correlations,  # Data
    cmap="coolwarm",  # Colormap
    vmin=-1,  # Min value to anchor the colormap
    vmax=1,  # Max value to anchor the colormap
    linewidths=0.5,  # Width of white lines
    xticklabels=True,  # Show column names of DataFrame on x-axis
    yticklabels=True,  # Show column names of DataFrame on y-axis
    annot=True,  # Show the value in addition to the color
)
ax.tick_params(left=False, bottom=False)  # Remove tick marks

## Resampling


In [ ]:
end_of_month = adj_close.resample("ME").last()
end_of_month.head()

In [ ]:
# Upsample from monthly to daily without transformation
end_of_month.resample("D").asfreq().head()

In [ ]:
# Upsample from monthly to weekly using forward filling
end_of_month.resample("W-FRI").ffill().head()

## Rolling Windows


In [ ]:
# Plot the moving average for MSFT with data from 2017
msft17 = msft.loc["2017", ["adj_close"]].copy()

# Add the 25 day moving average as a new column to the DataFrame
msft17["25day average"] = msft17["adj_close"].rolling(25).mean()
msft17.plot()